# Resource Check: Chronos · TimesFM · Moirai lokal

**Zweck.** Vor dem Aufbau des MoE-Frameworks klären, ob die drei Experten auf dieser
Maschine zero-shot lauffähig sind und ob RAM, Disk und Latenz für die geplanten
Experimente reichen.

Gemessen wird pro Experte:

| Grösse | Warum sie zählt |
|---|---|
| Ladezeit & Peak-RAM beim Laden | passen alle drei Experten gleichzeitig in den Speicher? |
| Checkpoint-Grösse auf Disk | Platzbedarf im HF-Cache |
| Inferenzlatenz (1 Serie, Batch) | hochgerechnet auf die Anzahl Serien × Horizonte × Fenster im Benchmark |
| Peak-RAM bei Inferenz | Batchgrösse, die wir uns leisten können |
| Output-Shape | liefert der Experte Quantile (für CRPS) oder nur Punktprognosen? |

Die Experten bleiben **eingefroren** — hier wird nichts trainiert, nur Forward-Passes.

## 1 · Umgebung und Hardware

In [1]:
import json, os, platform, shutil, subprocess, sys, time
import psutil

GIB = 1024 ** 3

def sysctl(key):
    try:
        return subprocess.check_output(["sysctl", "-n", key], text=True).strip()
    except Exception:
        return "n/a"

vm = psutil.virtual_memory()
disk = shutil.disk_usage(os.path.expanduser("~"))

env = {
    "Python": sys.version.split()[0],
    "Platform": f"{platform.system()} {platform.release()} ({platform.machine()})",
    "CPU": sysctl("machdep.cpu.brand_string"),
    "Kerne (logisch/physisch)": f"{psutil.cpu_count()} / {psutil.cpu_count(logical=False)}",
    "RAM total": f"{vm.total / GIB:.1f} GiB",
    "RAM frei (verfügbar)": f"{vm.available / GIB:.1f} GiB",
    "Disk frei (~)": f"{disk.free / GIB:.1f} GiB",
}
for k, v in env.items():
    print(f"{k:28s} {v}")

Python                       3.12.6
Platform                     Darwin 25.6.0 (arm64)
CPU                          Apple M1 Pro
Kerne (logisch/physisch)     10 / 10
RAM total                    16.0 GiB
RAM frei (verfügbar)         3.7 GiB
Disk frei (~)                66.4 GiB


In [2]:
import torch

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"torch                        {torch.__version__}")
print(f"MPS verfügbar / gebaut       {torch.backends.mps.is_available()} / {torch.backends.mps.is_built()}")
print(f"CUDA verfügbar               {torch.cuda.is_available()}")
print(f"-> DEVICE                    {DEVICE}")

# Apple Silicon: unified memory, d.h. MPS-Allokationen zählen gegen dieselben 16 GiB
# wie der CPU-RAM. Ein OOM auf MPS ist damit auch ein System-OOM-Risiko.
if DEVICE == "mps":
    os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # nicht-unterstützte Ops -> CPU

# Threads deterministisch halten, damit Latenzmessungen vergleichbar sind
torch.set_num_threads(psutil.cpu_count(logical=False) or 4)
print(f"torch threads                {torch.get_num_threads()}")

torch                        2.4.1
MPS verfügbar / gebaut       True / True
CUDA verfügbar               False
-> DEVICE                    mps
torch threads                10


In [3]:
# Versionen der Experten-Libraries (importlib.metadata, damit kein Modell geladen wird)
from importlib.metadata import PackageNotFoundError, version

for pkg in ["chronos-forecasting", "timesfm", "uni2ts", "gluonts",
            "transformers", "numpy", "pandas", "jax", "lightning"]:
    try:
        print(f"{pkg:22s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:22s} NICHT INSTALLIERT")

chronos-forecasting    2.3.2
timesfm                3.0.2
uni2ts                 2.0.0
gluonts                0.14.4
transformers           4.57.6
numpy                  1.26.4
pandas                 2.1.4
jax                    0.6.1
lightning              2.6.6


## 2 · Messharness

`probe(...)` misst Wall-Clock-Zeit und **Peak-RSS** eines Blocks. Peak-RSS wird von
einem Sampler-Thread erfasst (alle 20 ms), weil der RAM-Verbrauch beim Laden eines
Checkpoints kurzzeitig deutlich über dem Endzustand liegt (Datei -> Tensoren -> Kopie
aufs Device).

Alle Messungen landen in `RESULTS` und werden am Schluss zu einer Tabelle.

In [4]:
import gc, threading, traceback
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict

MB = 1024 ** 2
PROC = psutil.Process()
RESULTS = []          # Liste[dict] -> Zusammenfassungstabelle
LOADED = {}           # Experte -> geladenes Pipeline-Objekt (für Reuse in späteren Zellen)
ERRORS = {}           # Experte/Schritt -> Traceback-String


class _PeakSampler(threading.Thread):
    """Pollt RSS im Hintergrund und merkt sich das Maximum."""

    def __init__(self, interval=0.02):
        super().__init__(daemon=True)
        self.interval = interval
        self.peak = PROC.memory_info().rss
        self._done = threading.Event()   # NICHT _stop: Thread._stop ist intern belegt

    def run(self):
        while not self._done.is_set():
            try:
                self.peak = max(self.peak, PROC.memory_info().rss)
            except psutil.Error:
                break
            self._done.wait(self.interval)

    def stop(self):
        self._done.set()
        self.join(timeout=1.0)
        return self.peak


@contextmanager
def probe(expert, step, note=""):
    """Misst Zeit + Peak-RAM. Fehler werden protokolliert, nicht geworfen:
    ein fehlender Experte soll den Rest des Checks nicht abbrechen."""
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    rss_before = PROC.memory_info().rss
    sampler = _PeakSampler()
    sampler.start()
    t0 = time.perf_counter()
    status, err = "ok", None
    try:
        yield
    except BaseException as exc:                       # noqa: BLE001 - absichtlich breit
        status, err = "FEHLER", f"{type(exc).__name__}: {exc}"
        ERRORS[f"{expert}/{step}"] = traceback.format_exc()
    finally:
        dt = time.perf_counter() - t0
        peak = sampler.stop()
        rss_after = PROC.memory_info().rss
        rec = {
            "Experte": expert,
            "Schritt": step,
            "Status": status,
            "Zeit [s]": round(dt, 3),
            "Peak RSS [MB]": round(peak / MB, 1),
            "ΔRSS [MB]": round((rss_after - rss_before) / MB, 1),
            "Notiz": note or (err or ""),
        }
        RESULTS.append(rec)
        flag = "✓" if status == "ok" else "✗"
        print(f"{flag} {expert:16s} {step:22s} {dt:7.2f}s  peak {peak/MB:8.0f} MB  "
              f"Δ {(rss_after - rss_before)/MB:+8.0f} MB  {rec['Notiz']}")


def timeit_median(fn, repeats=3):
    """Median-Latenz über mehrere Läufe (erster Lauf = Warmup, verworfen)."""
    fn()                                              # Warmup: Lazy-Init, MPS-Kernel-Compile
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    ts.sort()
    return ts[len(ts) // 2]


def hf_cache_size():
    """Belegter Platz im HuggingFace-Cache, pro Repo."""
    from pathlib import Path
    root = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface")) / "hub"
    if not root.exists():
        return {}
    out = {}
    for repo in sorted(root.glob("models--*")):
        total = sum(f.stat().st_size for f in repo.rglob("*") if f.is_file() and not f.is_symlink())
        out[repo.name.replace("models--", "").replace("--", "/")] = round(total / MB, 1)
    return out


print(f"Basis-RSS dieses Prozesses: {PROC.memory_info().rss / MB:.0f} MB")

Basis-RSS dieses Prozesses: 202 MB


## 3 · Testdaten

Synthetischer Panel-Datensatz mit Trend + Saisonalität + Rauschen, plus eine
finanznahe Variante (Random Walk mit Volatilitäts-Clustering). Der Resource-Check
braucht keine echten Daten — entscheidend sind **Form und Grösse** der Eingabe:
`n_series × context_length`, wie im späteren Benchmark.

In [5]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(0)

CONTEXT = 512        # typische Kontextlänge der Experten
HORIZON = 64         # Prognosehorizont für den Check
N_SERIES = 32        # Panel-Breite für den Batch-Test


def make_seasonal(n_series, length, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(length)
    out = np.empty((n_series, length), dtype=np.float32)
    for i in range(n_series):
        period = rng.choice([7, 12, 24, 52])
        out[i] = (
            10.0
            + 0.01 * rng.normal(1.0, 0.5) * t                       # Trend
            + 3.0 * np.sin(2 * np.pi * t / period + rng.uniform(0, 6.28))  # Saison
            + rng.normal(0, 0.5, length)                            # Rauschen
        )
    return out


def make_financial(n_series, length, seed=1):
    """Log-Random-Walk mit GARCH-artigem Vola-Clustering - grob wie ein Kurs-Panel."""
    rng = np.random.default_rng(seed)
    out = np.empty((n_series, length), dtype=np.float32)
    for i in range(n_series):
        vol = np.empty(length)
        vol[0] = 0.01
        for t in range(1, length):
            vol[t] = np.sqrt(1e-6 + 0.9 * vol[t - 1] ** 2 + 0.05 * rng.normal() ** 2)
        out[i] = 100.0 * np.exp(np.cumsum(vol * rng.normal(size=length)))
    return out


PANEL_SEASONAL = make_seasonal(N_SERIES, CONTEXT)
PANEL_FINANCE = make_financial(N_SERIES, CONTEXT)
SERIES_1D = PANEL_SEASONAL[0]

print(f"Panel (saisonal):  {PANEL_SEASONAL.shape}  {PANEL_SEASONAL.nbytes / 1024:.0f} KB")
print(f"Panel (Finanz):    {PANEL_FINANCE.shape}")
print(f"Kontext {CONTEXT} -> Horizont {HORIZON}")

Panel (saisonal):  (32, 512)  64 KB
Panel (Finanz):    (32, 512)
Kontext 512 -> Horizont 64


## 4 · Device-Politik pro Experte

Auf Apple Silicon ist die Device-Wahl **nicht** einheitlich — das ist ein Ergebnis,
kein Implementierungsdetail:

| Experte | Device | Grund |
|---|---|---|
| Chronos-2 | `mps` möglich | reines PyTorch, MPS-kompatible Ops |
| TimesFM 2.5 | `cpu` | die Lib kennt nur `cuda`/`cpu`; erzwungenes MPS war in der Messung **langsamer** |
| Moirai 2.0 | `cpu` | `uni2ts` rechnet den Scaler intern in `float64` — das kann MPS grundsätzlich nicht |

Ausserdem: MPS nutzt auf dem M1 **denselben** Unified Memory wie die CPU. Ein
MPS-Puffer entlastet den RAM also nicht — das Speicherbudget bleibt geteilt.

In [6]:
# Ein float64-Feld aus GluonTS (past_is_pad) bricht MPS-Batches. Der Patch castet
# beim Batchify runter. Für den CPU-Pfad ist er ein No-op, deshalb global gesetzt.
import gluonts.torch.batchify as _batchify

if not getattr(_batchify, "_f64_patched", False):
    _orig_stack = _batchify.stack

    def _stack_f32(data, device=None):
        arr = np.asarray(data)
        if arr.dtype == np.float64:
            arr = arr.astype(np.float32)
        return _orig_stack(data=arr, device=device)

    _batchify.stack = _stack_f32
    _batchify._f64_patched = True

DEVICE_CHRONOS = DEVICE      # mps, wenn vorhanden
DEVICE_TIMESFM = "cpu"       # Lib-seitig keine MPS-Unterstützung
DEVICE_MOIRAI = "cpu"        # float64 im uni2ts-Scaler

print(f"Chronos  -> {DEVICE_CHRONOS}\nTimesFM  -> {DEVICE_TIMESFM}\nMoirai   -> {DEVICE_MOIRAI}")

Chronos  -> mps
TimesFM  -> cpu
Moirai   -> cpu


## 5 · Experte 1: Chronos-2 (Amazon)

`amazon/chronos-2` gibt Quantile direkt zurück — kein Sampling nötig, damit ist CRPS
ohne Monte-Carlo-Rauschen berechenbar. Rückgabe von `predict_quantiles`: eine Liste
pro Serie mit `(1, horizon, n_quantiles)` plus die Mittelwerte `(1, horizon)`.

In [7]:
CHRONOS_REPO = "amazon/chronos-2"
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

with probe("Chronos-2", "load", CHRONOS_REPO):
    from chronos import BaseChronosPipeline
    LOADED["chronos"] = BaseChronosPipeline.from_pretrained(
        CHRONOS_REPO, device_map=DEVICE_CHRONOS
    )
    print(f"    Pipeline: {type(LOADED['chronos']).__name__}, "
          f"Kontextlimit: {LOADED['chronos'].model_context_length}")

    Pipeline: Chronos2Pipeline, Kontextlimit: 8192
✓ Chronos-2        load                      1.43s  peak      449 MB  Δ      -95 MB  amazon/chronos-2


In [8]:
def chronos_forecast(panel, horizon=HORIZON):
    """Punkt- + Quantilprognose für ein (n_series, length)-Panel."""
    pipe = LOADED["chronos"]
    inputs = [torch.tensor(row) for row in panel]
    q, mean = pipe.predict_quantiles(inputs, prediction_length=horizon,
                                     quantile_levels=QUANTILES)
    return np.stack([m.numpy() for m in mean])[:, 0, :], np.stack([x.numpy() for x in q])[:, 0]


if "chronos" in LOADED:
    with probe("Chronos-2", "forecast 1 Serie"):
        pt, qt = chronos_forecast(PANEL_SEASONAL[:1])
        print(f"    point={pt.shape} quantiles={qt.shape}")

    with probe("Chronos-2", f"forecast {N_SERIES} Serien"):
        pt, qt = chronos_forecast(PANEL_SEASONAL)
        print(f"    point={pt.shape} quantiles={qt.shape}")

    with probe("Chronos-2", "forecast Finanzpanel"):
        pt_f, qt_f = chronos_forecast(PANEL_FINANCE)
        print(f"    point={pt_f.shape}  endlich={np.isfinite(pt_f).all()}")

    point=(1, 64) quantiles=(1, 64, 9)
✓ Chronos-2        forecast 1 Serie          0.57s  peak      511 MB  Δ     +109 MB  


    point=(32, 64) quantiles=(32, 64, 9)
✓ Chronos-2        forecast 32 Serien        0.42s  peak      617 MB  Δ     +100 MB  


    point=(32, 64)  endlich=True
✓ Chronos-2        forecast Finanzpanel      0.17s  peak      618 MB  Δ       +1 MB  


## 6 · Experte 2: TimesFM 2.5 (Google)

TimesFM muss vor dem ersten Forecast mit `compile(ForecastConfig(...))` auf
Kontext- und Horizontlänge festgelegt werden. **Für die Pipeline wichtig:**
`max_context`/`max_horizon` sind Teil des kompilierten Zustands — variable Horizonte
im Benchmark bedeuten entweder Neukompilierung oder Kompilieren auf das Maximum.

`use_continuous_quantile_head=True` liefert Quantile statt nur den Punktwert; die
Achse ist `[mean, q0.1, ..., q0.9]`, wir spalten den Mean ab.

**Fallstrick mit grosser Wirkung:** `per_core_batch_size` steht per Default auf `1`.
Damit rechnet TimesFM jede Serie einzeln und ist ~8x langsamer als nötig. Der Wert
wird beim `compile` fixiert und muss >= der gefütterten Batchgrösse sein.

In [9]:
TIMESFM_REPO = "google/timesfm-2.5-200m-pytorch"
TIMESFM_BATCH = 32      # >= grösster verwendeter Batch; beim compile fixiert

with probe("TimesFM-2.5", "load", TIMESFM_REPO):
    import timesfm
    LOADED["timesfm"] = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_REPO)

if "timesfm" in LOADED:
    with probe("TimesFM-2.5", "compile", f"ctx={CONTEXT} hor={HORIZON}"):
        LOADED["timesfm"].compile(timesfm.ForecastConfig(
            max_context=CONTEXT,
            max_horizon=HORIZON,
            normalize_inputs=True,
            use_continuous_quantile_head=True,
            fix_quantile_crossing=True,     # rohe Quantile sind leicht nicht-monoton
            per_core_batch_size=TIMESFM_BATCH,
        ))
        print(f"    inneres Modul auf Device: {LOADED['timesfm'].model.device}")

✓ TimesFM-2.5      load                      2.71s  peak     2261 MB  Δ     +778 MB  google/timesfm-2.5-200m-pytorch
    inneres Modul auf Device: cpu
✓ TimesFM-2.5      compile                   0.00s  peak     1397 MB  Δ       +0 MB  ctx=512 hor=64


In [10]:
def timesfm_forecast(panel, horizon=HORIZON):
    """TimesFMs Quantil-Achse hat 10 Einträge: [mean, q0.1, ..., q0.9].
    Index 0 wird abgespalten, damit das Raster zu Chronos/Moirai passt."""
    point, quant = LOADED["timesfm"].forecast(horizon=horizon, inputs=[row for row in panel])
    quant = np.asarray(quant)[..., 1:]          # -> (n, horizon, 9)
    return np.asarray(point), quant


if "timesfm" in LOADED:
    with probe("TimesFM-2.5", "forecast 1 Serie"):
        pt, qt = timesfm_forecast(PANEL_SEASONAL[:1])
        print(f"    point={pt.shape} quantiles={qt.shape}")

    with probe("TimesFM-2.5", f"forecast {N_SERIES} Serien"):
        pt, qt = timesfm_forecast(PANEL_SEASONAL)
        print(f"    point={pt.shape} quantiles={qt.shape}")

    with probe("TimesFM-2.5", "forecast Finanzpanel"):
        pt_f, _ = timesfm_forecast(PANEL_FINANCE)
        print(f"    point={pt_f.shape}  endlich={np.isfinite(pt_f).all()}")

    point=(1, 64) quantiles=(1, 64, 9)
✓ TimesFM-2.5      forecast 1 Serie          0.64s  peak     1752 MB  Δ     +374 MB  


    point=(32, 64) quantiles=(32, 64, 9)
✓ TimesFM-2.5      forecast 32 Serien        0.65s  peak     1789 MB  Δ      +17 MB  


    point=(32, 64)  endlich=True
✓ TimesFM-2.5      forecast Finanzpanel      0.61s  peak     1806 MB  Δ      +18 MB  


## 7 · Experte 3: Moirai 2.0 (Salesforce)

Moirai kommt über einen **GluonTS-Predictor**, nicht über eine Array-API. Das ist der
umständlichste der drei Experten: Eingabe muss ein GluonTS-Dataset sein, und
`prediction_length`/`context_length` stehen im Konstruktor — ein anderer Horizont
heisst neues `Moirai2Forecast`-Objekt (das Modul selbst wird aber wiederverwendet,
also ohne erneuten Download).

Rückgabe ist `QuantileForecast` mit `forecast_array` der Form `(9, horizon)`.

In [11]:
MOIRAI_REPO = "Salesforce/moirai-2.0-R-small"

with probe("Moirai-2.0", "load", MOIRAI_REPO):
    from gluonts.dataset.common import ListDataset
    from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
    LOADED["moirai_module"] = Moirai2Module.from_pretrained(MOIRAI_REPO)
    n_params = sum(p.numel() for p in LOADED["moirai_module"].parameters())
    print(f"    Parameter: {n_params / 1e6:.1f}M")

    Parameter: 11.4M
✓ Moirai-2.0       load                      0.43s  peak     1840 MB  Δ       +2 MB  Salesforce/moirai-2.0-R-small


In [12]:
_MOIRAI_PREDICTORS = {}   # (horizon, context, batch) -> PyTorchPredictor


def moirai_predictor(horizon=HORIZON, context=CONTEXT, batch_size=N_SERIES):
    """Predictor cachen: Neuaufbau pro Horizont ist billig, aber nicht gratis."""
    key = (horizon, context, batch_size)
    if key not in _MOIRAI_PREDICTORS:
        fc = Moirai2Forecast(
            module=LOADED["moirai_module"],
            prediction_length=horizon,
            context_length=context,
            target_dim=1,
            feat_dynamic_real_dim=0,
            past_feat_dynamic_real_dim=0,
        )
        _MOIRAI_PREDICTORS[key] = fc.create_predictor(batch_size=batch_size,
                                                      device=DEVICE_MOIRAI)
    return _MOIRAI_PREDICTORS[key]


def moirai_forecast(panel, horizon=HORIZON, freq="D"):
    pred = moirai_predictor(horizon=horizon, batch_size=max(len(panel), 1))
    ds = ListDataset([{"start": "2020-01-01", "target": row} for row in panel], freq=freq)
    forecasts = list(pred.predict(ds))
    # forecast_array ist (9, horizon) -> transponieren auf (horizon, 9) wie bei den anderen
    quant = np.stack([f.forecast_array.T for f in forecasts])      # (n, horizon, 9)
    point = np.stack([f.quantile(0.5) for f in forecasts])         # (n, horizon)
    return point, quant


if "moirai_module" in LOADED:
    with probe("Moirai-2.0", "forecast 1 Serie"):
        pt, qt = moirai_forecast(PANEL_SEASONAL[:1])
        print(f"    point={pt.shape} quantiles={qt.shape}")

    with probe("Moirai-2.0", f"forecast {N_SERIES} Serien"):
        pt, qt = moirai_forecast(PANEL_SEASONAL)
        print(f"    point={pt.shape} quantiles={qt.shape}")

    with probe("Moirai-2.0", "forecast Finanzpanel"):
        pt_f, _ = moirai_forecast(PANEL_FINANCE)
        print(f"    point={pt_f.shape}  endlich={np.isfinite(pt_f).all()}")

    point=(1, 64) quantiles=(1, 64, 9)
✓ Moirai-2.0       forecast 1 Serie          0.03s  peak     1810 MB  Δ       +1 MB  


    point=(32, 64) quantiles=(32, 64, 9)
✓ Moirai-2.0       forecast 32 Serien        0.07s  peak     1810 MB  Δ       +0 MB  


    point=(32, 64)  endlich=True
✓ Moirai-2.0       forecast Finanzpanel      0.07s  peak     1810 MB  Δ       +0 MB  


## 8 · Der eigentliche Test: alle drei Experten gleichzeitig im Speicher

Für das MoE brauchen wir alle Experten **parallel** geladen — das Gating-Netz
kombiniert ihre Ausgaben pro Serie. Entscheidend ist also nicht der Peak eines
einzelnen Modells, sondern der Gesamtverbrauch.

In [13]:
present = [k for k in ("chronos", "timesfm", "moirai_module") if k in LOADED]
print(f"Gleichzeitig geladen: {present}")
print(f"RSS jetzt: {PROC.memory_info().rss / MB:.0f} MB "
      f"({PROC.memory_info().rss / GIB:.2f} GiB von {vm.total / GIB:.0f} GiB)\n")


def all_experts(panel, horizon=HORIZON):
    """Ein Forward-Pass durch alle verfügbaren Experten -> einheitliches Format."""
    out = {}
    if "chronos" in LOADED:
        out["Chronos-2"] = chronos_forecast(panel, horizon)
    if "timesfm" in LOADED:
        out["TimesFM-2.5"] = timesfm_forecast(panel, horizon)
    if "moirai_module" in LOADED:
        out["Moirai-2.0"] = moirai_forecast(panel, horizon)
    return out


with probe("ALLE", f"Ensemble-Pass {N_SERIES} Serien"):
    preds = all_experts(PANEL_SEASONAL)
    for name, (pt, qt) in preds.items():
        print(f"    {name:14s} point {str(pt.shape):12s} quantiles {qt.shape}")

Gleichzeitig geladen: ['chronos', 'timesfm', 'moirai_module']
RSS jetzt: 1810 MB (1.77 GiB von 16 GiB)



    Chronos-2      point (32, 64)     quantiles (32, 64, 9)
    TimesFM-2.5    point (32, 64)     quantiles (32, 64, 9)
    Moirai-2.0     point (32, 64)     quantiles (32, 64, 9)
✓ ALLE             Ensemble-Pass 32 Serien    1.27s  peak     1810 MB  Δ      -60 MB  


In [14]:
# Einheitliches Ausgabeformat prüfen: lassen sich die Experten zu einem Tensor
# stapeln? Das ist die Voraussetzung für jede Kombinationsstrategie.
if len(preds) > 1:
    names = list(preds)
    stacked_point = np.stack([preds[n][0] for n in names])     # (n_experts, n_series, horizon)
    print(f"Gestapelte Punktprognosen: {stacked_point.shape}  (Experten, Serien, Horizont)")

    # Baseline "einfacher Durchschnitt" - hier nur als Formatnachweis, nicht als Evaluation
    simple_avg = stacked_point.mean(axis=0)
    print(f"Durchschnitts-Baseline:    {simple_avg.shape}")

    # Alle drei liefern nach der Normalisierung dasselbe Raster: 9 Levels 0.1..0.9
    for n in names:
        print(f"  {n:14s} Quantile: {preds[n][1].shape}  (Serien, Horizont, Levels)")
    stacked_q = np.stack([preds[n][1] for n in names])
    print(f"\nGestapelte Quantile:       {stacked_q.shape}  "
          "(Experten, Serien, Horizont, Levels)")
    print(f"Level-Set einheitlich:     {QUANTILES}")
    print("-> CRPS kann direkt auf diesem gemeinsamen Raster berechnet werden.")

Gestapelte Punktprognosen: (3, 32, 64)  (Experten, Serien, Horizont)
Durchschnitts-Baseline:    (32, 64)
  Chronos-2      Quantile: (32, 64, 9)  (Serien, Horizont, Levels)
  TimesFM-2.5    Quantile: (32, 64, 9)  (Serien, Horizont, Levels)
  Moirai-2.0     Quantile: (32, 64, 9)  (Serien, Horizont, Levels)

Gestapelte Quantile:       (3, 32, 64, 9)  (Experten, Serien, Horizont, Levels)
Level-Set einheitlich:     [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
-> CRPS kann direkt auf diesem gemeinsamen Raster berechnet werden.


## 9 · Skalierung: reicht die Latenz für den geplanten Benchmark?

Hochrechnung von einer Messung auf das volle Experiment. Wir messen die Latenz über
Batchgrössen und extrapolieren auf `n_series × n_windows × n_horizons`.

In [15]:
BATCH_SIZES = [1, 8, 32]
scaling = []

for bs in BATCH_SIZES:
    panel = PANEL_SEASONAL[:bs] if bs <= N_SERIES else make_seasonal(bs, CONTEXT, seed=7)
    for name, fn in [("Chronos-2", chronos_forecast if "chronos" in LOADED else None),
                     ("TimesFM-2.5", timesfm_forecast if "timesfm" in LOADED else None),
                     ("Moirai-2.0", moirai_forecast if "moirai_module" in LOADED else None)]:
        if fn is None:
            continue
        try:
            dt = timeit_median(lambda: fn(panel), repeats=3)
            scaling.append({"Experte": name, "Batch": bs, "Zeit [s]": round(dt, 4),
                            "ms/Serie": round(1000 * dt / bs, 2)})
        except Exception as exc:                          # noqa: BLE001
            ERRORS[f"{name}/scaling bs={bs}"] = traceback.format_exc()
            scaling.append({"Experte": name, "Batch": bs, "Zeit [s]": float("nan"),
                            "ms/Serie": float("nan")})

SCALING = pd.DataFrame(scaling)
display(SCALING.pivot(index="Batch", columns="Experte", values="ms/Serie")
        .rename_axis(columns="ms pro Serie"))

ms pro Serie,Chronos-2,Moirai-2.0,TimesFM-2.5
Batch,,,
1,50.72,14.41,537.30
8,7.69,4.97,73.25
32,5.37,2.73,18.55


In [16]:
# Hochrechnung auf ein realistisches Benchmark-Budget
N_BENCH_SERIES = 1000     # Serien über alle Datensätze
N_WINDOWS = 5             # Rolling-Origin-Fenster
N_HORIZONS = 3            # z.B. kurz / mittel / lang

total_calls = N_BENCH_SERIES * N_WINDOWS * N_HORIZONS
print(f"Budget: {N_BENCH_SERIES} Serien x {N_WINDOWS} Fenster x {N_HORIZONS} Horizonte "
      f"= {total_calls:,} Serien-Prognosen pro Experte\n")

usable = SCALING.dropna(subset=["ms/Serie"])
best = usable.loc[usable.groupby("Experte")["ms/Serie"].idxmin()]
grand_total = 0.0
for _, row in best.iterrows():
    secs = total_calls * row["ms/Serie"] / 1000
    grand_total += secs
    print(f"  {row['Experte']:14s} @ Batch {int(row['Batch']):>3d}: "
          f"{row['ms/Serie']:7.2f} ms/Serie -> {secs / 60:7.1f} min")

print(f"\n  {'ALLE drei':14s} {'':16s} {'':7s} -> {grand_total / 60:7.1f} min "
      f"({grand_total / 3600:.1f} h) für einen kompletten Durchlauf")
print("\nHinweis: Einmal berechnete Expertenprognosen sollten gecacht werden - die "
      "Kombinationsstrategien werden auf denselben Vorhersagen trainiert und "
      "brauchen keinen erneuten Forward-Pass.")

Budget: 1000 Serien x 5 Fenster x 3 Horizonte = 15,000 Serien-Prognosen pro Experte

  Chronos-2      @ Batch  32:    5.37 ms/Serie ->     1.3 min
  Moirai-2.0     @ Batch  32:    2.73 ms/Serie ->     0.7 min
  TimesFM-2.5    @ Batch  32:   18.55 ms/Serie ->     4.6 min

  ALLE drei                               ->     6.7 min (0.1 h) für einen kompletten Durchlauf

Hinweis: Einmal berechnete Expertenprognosen sollten gecacht werden - die Kombinationsstrategien werden auf denselben Vorhersagen trainiert und brauchen keinen erneuten Forward-Pass.


## 10 · Speicherplatz

In [17]:
cache = hf_cache_size()
relevant = {k: v for k, v in cache.items()
            if any(t in k.lower() for t in ("chronos", "timesfm", "moirai"))}

for repo, mb in sorted(relevant.items(), key=lambda kv: -kv[1]):
    print(f"  {repo:44s} {mb:8.0f} MB")
print(f"  {'-' * 44} {'-' * 8}")
print(f"  {'Summe Experten-Checkpoints':44s} {sum(relevant.values()):8.0f} MB")
print(f"\nDisk frei: {shutil.disk_usage(os.path.expanduser('~')).free / GIB:.1f} GiB")

  google/timesfm-2.5-200m-pytorch                   882 MB
  amazon/chronos-2                                  456 MB
  Salesforce/moirai-2.0-R-small                      43 MB
  -------------------------------------------- --------
  Summe Experten-Checkpoints                       1382 MB

Disk frei: 66.4 GiB


## 11 · Zusammenfassung

In [18]:
SUMMARY = pd.DataFrame(RESULTS)
display(SUMMARY)

if ERRORS:
    print(f"\n{len(ERRORS)} Fehler protokolliert:")
    for key in ERRORS:
        print(f"  - {key}")
    print("\nVollen Traceback abrufen mit: print(ERRORS['<key>'])")
else:
    print("\nKeine Fehler.")

,Experte,Schritt,Status,Zeit [s],Peak RSS [MB],ΔRSS [MB],Notiz
0,Chronos-2,load,ok,1.431,448.9,-95.0,amazon/chronos-2
1,Chronos-2,forecast 1 Serie,ok,0.569,511.1,108.8,
2,Chronos-2,forecast 32 Serien,ok,0.418,617.2,100.3,
3,Chronos-2,forecast Finanzpanel,ok,0.172,618.0,0.7,
4,TimesFM-2.5,load,ok,2.710,2261.0,778.0,google/timesfm-2.5-200m-pytorch
5,TimesFM-2.5,compile,ok,0.000,1397.5,0.0,ctx=512 hor=64
6,TimesFM-2.5,forecast 1 Serie,ok,0.636,1751.7,374.2,
7,TimesFM-2.5,forecast 32 Serien,ok,0.648,1788.6,16.8,
8,TimesFM-2.5,forecast Finanzpanel,ok,0.613,1806.2,17.6,
9,Moirai-2.0,load,ok,0.433,1840.4,2.0,Salesforce/moirai-2.0-R-small



Keine Fehler.


In [19]:
# Verdikt gegen das RAM-Budget
peak_all = SUMMARY["Peak RSS [MB]"].max()
rss_now = PROC.memory_info().rss / MB
headroom = (vm.total / MB) - peak_all
EXPERT_NAMES = {"Chronos-2", "TimesFM-2.5", "Moirai-2.0"}
ok_experts = set(SUMMARY.query("Status == 'ok'")["Experte"]) & EXPERT_NAMES
n_ok = len(ok_experts)

print(f"Experten lauffähig               {n_ok} / 3")
print(f"Peak RSS über alle Schritte      {peak_all / 1024:.2f} GiB")
print(f"RSS mit allen drei geladen       {rss_now / 1024:.2f} GiB")
print(f"System-RAM                       {vm.total / GIB:.0f} GiB")
print(f"Headroom zum Peak                {headroom / 1024:.2f} GiB "
      f"({100 * headroom / (vm.total / MB):.0f} %)")
print()
if n_ok == 3 and headroom / 1024 > 4:
    print("VERDIKT: Ressourcen reichen. Alle drei Experten laufen gleichzeitig "
          "mit komfortablem Headroom.")
elif n_ok == 3:
    print("VERDIKT: Läuft, aber der Headroom ist knapp - Batchgrössen begrenzen "
          "und Expertenprognosen auf Disk cachen.")
else:
    print("VERDIKT: Nicht alle Experten lauffähig - siehe ERRORS.")

Experten lauffähig               3 / 3
Peak RSS über alle Schritte      2.21 GiB
RSS mit allen drei geladen       1.96 GiB
System-RAM                       16 GiB
Headroom zum Peak                13.79 GiB (86 %)

VERDIKT: Ressourcen reichen. Alle drei Experten laufen gleichzeitig mit komfortablem Headroom.


In [20]:
# Ergebnisse für den Bericht sichern
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
out = ROOT / "results"
out.mkdir(exist_ok=True)
SUMMARY.to_csv(out / "resource_check_summary.csv", index=False)
SCALING.to_csv(out / "resource_check_scaling.csv", index=False)
with open(out / "resource_check_env.json", "w") as fh:
    json.dump({"env": env, "device": {"chronos": DEVICE_CHRONOS,
                                      "timesfm": DEVICE_TIMESFM,
                                      "moirai": DEVICE_MOIRAI},
               "checkpoints_mb": relevant}, fh, indent=2)
print(f"Geschrieben nach {out.resolve()}/")

Geschrieben nach /Users/sebastianjung/Documents/HSG/HS 26/BP/Prototype/results/


## 12 · Was der Check für das Projekt bedeutet

**Offene Punkte für die Besprechung mit Lukas:**

1. **Versionen festnageln.** Hier laufen Chronos-2, TimesFM-2.5-200M und
   Moirai-2.0-R-small. Das sind jeweils die aktuellen Generationen, nicht die in den
   Originalpapern evaluierten (Chronos-Bolt, TimesFM-1.0, Moirai-1.1-R). Für
   Vergleichbarkeit mit publizierten Zahlen wären die älteren Versionen relevant.
2. **Quantilraster vereinheitlichen.** Die Experten liefern unterschiedliche
   Quantil-Achsen (9 vs. 10). Das MoE braucht ein gemeinsames Level-Set.
3. **Leakage prüfen.** Alle drei Checkpoints sind auf grossen Zeitreihenkorpora
   vortrainiert; die Trainingskorpora überlappen mit gängigen Benchmarks. Für eine
   faire Zero-Shot-Evaluation muss die Überlappung dokumentiert werden.
4. **Prognosen cachen.** Die Kombinationsstrategien (Durchschnitt, statische
   Gewichte, Gating-Netz) trainieren alle auf denselben Expertenausgaben. Einmal
   rechnen, auf Disk legen, dann nur noch das Gating trainieren.